<a href="https://colab.research.google.com/github/marinazakimi/ECAA08--Grupo-06/blob/main/etapa-01-logica/03_Tautologias_e_Contradicoes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prática 03: Validação Formal de Intertravamentos via Python

Nessa aula, vamos utilizar Python para validar as regras lógicas de segurança do Drone Agrícola. O objetivo é simular a **Intertrava de Trip de Emergência** e provar computacionalmente que o **Teorema de Segurança Funcional** (impossibilidade do estado de perigo) é uma contradição lógica.

In [1]:
import itertools
import pandas as pd

# 1. Simulação da Intertrava de Trip de Emergência
print("--- ANÁLISE DE TRIP DE EMERGÊNCIA ---")

# Possíveis estados das variáveis de entrada (0 = Normal, 1 = Falha/Acionado)
# g1: Perda GPS | b1: Bateria Crítica | w1: Vento Severo | e1: Botão de Emergência
entradas = list(itertools.product([False, True], repeat=4))
resultados = []

for g1, b1, w1, e1 in entradas:
    # Lógica Combinacional de Falha (F)
    F = g1 or b1 or w1 or e1

    # Ações de Saída baseadas no Intertravamento (Se F for True, aplica restrições)
    p1 = not F  # Bomba de calda (True = Ligada, False = Desligada)
    m1 = not F  # Motores (True = Armados, False = Desarmados)
    a1 = F      # Alarme na IHM (True = Ativado, False = Desativado)

    resultados.append([g1, b1, w1, e1, F, p1, m1, a1])

# Visualização da Tabela Verdade usando Pandas
colunas = ['GPS_Falha (g1)', 'Bat_Critica (b1)', 'Vento (w1)', 'Emergencia (e1)',
           'TRIP (F)', 'Bomba (p1)', 'Motores (m1)', 'Alarme (a1)']
df_trip = pd.DataFrame(resultados, columns=colunas)

# Mostrando os casos críticos onde o botão de emergência foi acionado
display(df_trip[df_trip['Emergencia (e1)'] == True].head(4))

--- ANÁLISE DE TRIP DE EMERGÊNCIA ---


,GPS_Falha (g1),Bat_Critica (b1),Vento (w1),Emergencia (e1),TRIP (F),Bomba (p1),Motores (m1),Alarme (a1)
1,False,False,False,True,True,False,False,True
3,False,False,True,True,True,False,False,True
5,False,True,False,True,True,False,False,True
7,False,True,True,True,True,False,False,True


**Prova de Invariante de Segurança (Estado de Perigo Catastrófico)**

Conforme definido na teoria, o estado de perigo ($S_{\text{perigo}}$) ocorre se o botão de emergência estiver acionado ($e_1$) E os motores continuarem armados ($m_1$).
A controladora possui a regra de hardware: $e_1 \rightarrow \neg m_1$.

Vamos provar através de uma Tabela Verdade que a ocorrência simultânea da regra E do estado de perigo resulta em uma **Contradição** (sempre Falso).

In [2]:
# 2. Prova do Teorema de Segurança Funcional
print("\n--- PROVA DE CONTRADIÇÃO (ESTADO DE PERIGO) ---")

combinacoes_seguranca = list(itertools.product([False, True], repeat=2))
prova = []

for e1, m1 in combinacoes_seguranca:
    # Estado de Perigo: Botão acionado E motores girando
    S_perigo = e1 and m1

    # Regra de Intertravamento (e1 -> ~m1): Equivalente a (~e1 or ~m1)
    regra_interlock = (not e1) or (not m1)

    # Teorema: O Sistema pode atingir o estado de perigo operando sob a regra?
    teorema_phi = S_perigo and regra_interlock

    prova.append([e1, m1, S_perigo, regra_interlock, teorema_phi])

df_prova = pd.DataFrame(prova, columns=['Emergencia (e1)', 'Motor_Armado (m1)',
                                        'Estado de Perigo', 'Regra Ativa', 'Sistema em Falha (Phi)'])

display(df_prova)

# Verificação computacional da Tautologia/Contradição
if df_prova['Sistema em Falha (Phi)'].any():
    print("ALERTA: O sistema não é seguro. Falha lógica encontrada.")
else:
    print("SUCESSO: A expressão Phi é uma CONTRADIÇÃO formal. O estado de perigo nunca ocorrerá sob esta regra.")


--- PROVA DE CONTRADIÇÃO (ESTADO DE PERIGO) ---


,Emergencia (e1),Motor_Armado (m1),Estado de Perigo,Regra Ativa,Sistema em Falha (Phi)
0,False,False,False,True,False
1,False,True,False,True,False
2,True,False,False,True,False
3,True,True,True,False,False


SUCESSO: A expressão Phi é uma CONTRADIÇÃO formal. O estado de perigo nunca ocorrerá sob esta regra.
